# Stock Research Agent

Give it a ticker and it writes a short research report. A planner call looks at a quick company profile and picks which data tools to run (prices, financials, valuation, earnings) and how much price history to pull. The tools pull from yfinance and the headlines come from NewsAPI. Each headline goes through classify → extract → summarize and gets routed to one of three analysts (market, financial, or news). Irrelevant ones stop after classify. A writer turns the findings into the report.

After that an evaluator grades it against a checklist and we revise a few times if needed. Failed checks from the first draft get saved to a json file and fed into the next run.

Setup: put `OPENAI_API_KEY` and `NEWS_API_KEY` in Colab secrets and run top to bottom. All calls use JSON mode. The evaluator runs on gpt-4o and everything else on gpt-4o-mini, which keeps it cheaper and faster. The memory test at the end makes a lot of API calls (6 full runs) so it takes a while.

## Agents

Each agent is its own function so it's easy to find. `run_research` at the bottom is the coordinator that calls them in order.

- Planner (`planner_agent`): looks at the company profile and picks which data tools to run and how much price history to pull
- Article agent (`article_agent`): runs the prompt chain on each headline, `classify_article` → `extract_facts` → `summarize_facts`
- Router (`route_articles`): sends each article to the analyst that handles its category
- Specialist analysts (`analyst_agent`, defined in `SPECIALISTS`): market, financial, and news; each one writes 2–3 findings from its own tool data and articles
- Writer (`writer_agent`): turns the findings into the report
- Evaluator (`evaluator_agent`): grades the report against the checklist
- Optimizer (`optimizer_agent`): edits the report to fix the checks it failed

The tools (`TOOLS`, run through `run_tool`) and memory (`load_lessons` / `save_lessons`) aren't agents. The planner decides which tools run, and the lessons get passed to the planner, analysts, and writer.

In [ ]:
import os, re, json
from datetime import datetime, timedelta
import requests
import yfinance as yf
from openai import OpenAI
from google.colab import userdata

client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))
NEWS_API_KEY = userdata.get("NEWS_API_KEY")


# every LLM call goes through here (json mode, so the reply can be indexed right away)
def call_llm(prompt, temp=0.2, model="gpt-4o-mini"):
    resp = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=temp,
        response_format={"type": "json_object"},
    )
    return json.loads(resp.choices[0].message.content)

## Data

NewsAPI for headlines from the last 3 weeks. The query uses the company name or ticker plus a few finance words to avoid random hits. Everything else comes from yfinance through the tools below.

Preprocessing just cleans whitespace, drops duplicates, and filters out the "[Removed]" articles NewsAPI sometimes returns.

In [ ]:
# headlines from the last 3 weeks
def get_news(stock):
    name = stock.info["longName"]
    # name or ticker plus a few finance words, otherwise we get a lot of random hits
    params = {
        "q": f'("{name}" OR "{stock.ticker}") AND (earnings OR stock OR revenue OR shares OR market)',
        "from": (datetime.now() - timedelta(days=21)).strftime("%Y-%m-%d"),
        "sortBy": "relevancy",
        "pageSize": 10,
        "language": "en",
        "apiKey": NEWS_API_KEY,
    }
    res = requests.get("https://newsapi.org/v2/everything", params=params).json()
    if res["status"] != "ok":
        print("NewsAPI error:", res["message"])

    articles = []
    for a in res.get("articles", []):
        articles.append(f"{a['title']}. {a['description'] or ''}")
    return articles


# clean whitespace, drop duplicates and skip deleted ones (NewsAPI sends those back as "[Removed]")
def preprocess(articles):
    clean = []
    for text in articles:
        text = " ".join(text.split())
        if text not in clean and "[Removed]" not in text:
            clean.append(text)
    return clean

## Tools

Each data tool is a small function, and `TOOLS` is the menu the planner picks from (it only sees the descriptions). All the numbers are computed here so the model never has to do any math, it only interprets them.

- prices: latest close, total return over the period, annualized volatility, and max drawdown
- financials: revenue and its growth, net income and margin, and free cash flow from the latest annual statements, plus debt-to-equity and current ratio from the latest quarterly balance sheet. Both dates are included so the report can say which year and quarter the numbers are from
- valuation: trailing and forward P/E
- earnings: last quarter's EPS vs the analyst estimate, and the surprise %

`run_tool` runs one tool by name. If a tool errors out (like a company Yahoo has no P/E for), it prints the error and the run keeps going without that data.

A few fields (period, dates, latest close, absolute revenue and net income, raw EPS) are skipped later when building per-metric checks because they're either just labels or raw levels.

In [ ]:
# the planner picks from these (it only ever sees the descriptions)
TOOLS = {
    "prices": "price return, volatility and max drawdown over the period you pick",
    "financials": "revenue growth, margins, free cash flow, debt and liquidity",
    "valuation": "trailing and forward P/E",
    "earnings": "last quarter's EPS compared to the analyst estimate",
}


# dropna because today's row can be NaN while the market is still open
def get_prices(stock, period):
    close = stock.history(period=period)["Close"].dropna()
    returns = close.pct_change().dropna()
    return {
        "period": period,
        "latest_close_usd": round(close.iloc[-1], 2),
        "price_return_pct": round((close.iloc[-1] / close.iloc[0] - 1) * 100, 1),
        "annualized_volatility_pct": round(returns.std() * 252 ** 0.5 * 100, 1),
        "max_drawdown_pct": round((close / close.cummax() - 1).min() * 100, 1),
    }


# latest annual results (column 0 = most recent year), plus debt and liquidity
# from the latest quarterly balance sheet
def get_financials(stock):
    inc = stock.income_stmt
    bal = stock.quarterly_balance_sheet
    rev = inc.loc["Total Revenue"]
    ni = inc.loc["Net Income"].iloc[0]
    return {
        "fiscal_year_end": str(inc.columns[0].date()),
        "revenue_billion_usd": round(rev.iloc[0] / 1e9, 1),
        "revenue_growth_pct": round((rev.iloc[0] / rev.iloc[1] - 1) * 100, 1),
        "net_income_billion_usd": round(ni / 1e9, 1),
        "net_margin_pct": round(ni / rev.iloc[0] * 100, 1),
        "balance_sheet_date": str(bal.columns[0].date()),
        "debt_to_equity": round(bal.loc["Total Debt"].iloc[0] / bal.loc["Stockholders Equity"].iloc[0], 2),
        "current_ratio": round(bal.loc["Current Assets"].iloc[0] / bal.loc["Current Liabilities"].iloc[0], 2),
        "free_cash_flow_billion_usd": round(stock.cashflow.loc["Free Cash Flow"].iloc[0] / 1e9, 1),
    }


# P/E ratios from yahoo's quote data
def get_valuation(stock):
    info = stock.info
    return {"trailing_pe": round(info["trailingPE"], 1), "forward_pe": round(info["forwardPE"], 1)}


# last reported quarter: actual EPS vs what analysts expected
def get_earnings(stock):
    hist = stock.earnings_history.sort_index()
    last = hist.iloc[-1]
    return {
        "quarter_end": str(hist.index[-1].date()),
        "eps_estimate": round(last["epsEstimate"], 2),
        "eps_actual": round(last["epsActual"], 2),
        "eps_surprise_pct": round((last["epsActual"] / last["epsEstimate"] - 1) * 100, 1),
    }


# runs one tool by name. if it errors out (like yahoo having no P/E for a company)
# we print it and return {} so the run keeps going without that data
def run_tool(name, stock, plan):
    try:
        if name == "prices":
            return get_prices(stock, plan["period"])
        elif name == "financials":
            return get_financials(stock)
        elif name == "valuation":
            return get_valuation(stock)
        elif name == "earnings":
            return get_earnings(stock)
    except Exception as e:
        print(f"{name} tool failed, skipping it: {e}")
    return {}

## Planner

First step of every run. The planner agent gets a quick company profile (name, sector, industry) plus any lessons from earlier runs. It picks three of the four data tools and how much price history to pull ('6mo', '1y', or '2y'). Only getting three tools means it has to decide what matters most for that company. News always runs (last 3 weeks) since it feeds the prompt chain.

If the period isn't one we support, it falls back to '1y'. Tool names don't always come back clean, so we just search the reply as a string and ignore anything that isn't a real tool.

In [ ]:
# planner: looks at the company profile and decides which tools to run
def planner_agent(ticker, profile, lessons):
    prompt = (
        f"We're writing a short research report on {ticker} that covers price behavior, financial health, "
        f"and recent news. Company: {profile}\n"
        f"Data tools we can run: {TOOLS}\n"
        "Pick the three tools that matter most for this company right now and how much price history "
        "to use ('6mo', '1y' or '2y').\n"
        f"Things earlier reports missed: {lessons or 'nothing yet'}\n"
        "Reply in JSON with 'steps' (a list of tool names), 'period', and 'reason'."
    )
    plan = call_llm(prompt)

    # tool names don't always come back clean, so search the reply as a string and only
    # keep real tools. anything other than 6mo/1y/2y falls back to 1y
    plan["steps"] = [t for t in TOOLS if t in str(plan["steps"]).lower()]
    if plan["period"] not in ["6mo", "1y", "2y"]:
        plan["period"] = "1y"
    return plan

## Article agent (prompt chain)

Each headline goes through a short prompt chain in `article_agent`: classify > extract > summarize. Each step is its own function and its own LLM call. Classify picks one of earnings, market, news, or irrelevant. Anything marked irrelevant stops after the first call. For the others, extract pulls key facts focused on that category, and summarize turns those facts into one investor-oriented sentence. The summary never sees the original article text, only the extracted facts.

The same category is later used for routing: earnings articles go to the financial analyst, market to the market analyst, and news to the news analyst (see `route_articles` below).

In [ ]:
# article agent = prompt chain: classify -> extract -> summarize

# what the extract step should focus on for each category
EXTRACT_FOCUS = {
    "earnings": "reported results, guidance, and beats or misses",
    "market": "price moves, analyst ratings or price targets, and trading activity",
    "news": "the event, product, or deal, and who is involved",
}

# step 1: pick a category
def classify_article(text, ticker):
    instruction = (
        f"Which category fits this article about {ticker}?\n"
        "- earnings: results, guidance, EPS, revenue\n"
        "- market: stock price moves, analyst price targets, trading volume\n"
        "- news: other company events like products, deals, lawsuits, or people\n"
        f"- irrelevant: not really about {ticker}\n\n"
        f"Article: {text[:600]}\n\n"
        "Reply in JSON with a 'category' key."
    )
    return call_llm(instruction, 0)["category"].lower()

# step 2: pull out the key facts for that category
def extract_facts(text, ticker, cat):
    instruction = (
        f"List the key facts in this {cat} article about {ticker}, mainly {EXTRACT_FOCUS[cat]}. "
        "Keep numbers exactly as written and don't add anything that isn't in the article.\n\n"
        f"Article: {text[:600]}\n\n"
        "Reply in JSON with a 'facts' key (a list of short strings)."
    )
    return call_llm(instruction, 0)["facts"]

# step 3: one sentence for an investor. it only gets the facts, never the article itself
def summarize_facts(facts):
    instruction = (f"Summarize these facts in one sentence for an investor: {facts}\n"
                   "Reply in JSON with a 'summary' key.")
    return call_llm(instruction, 0)["summary"]

# whole chain on one headline, irrelevant ones stop after classify
def article_agent(text, ticker):
    cat = classify_article(text, ticker)
    article = {"text": text, "category": cat}
    if cat in EXTRACT_FOCUS:
        article["facts"] = extract_facts(text, ticker, cat)
        article["summary"] = summarize_facts(article["facts"])
    return article

## Specialist analysts + router

`SPECIALISTS` defines the three analyst agents in one place: what each one focuses on, which tools' data it gets, and which article categories it handles. The market analyst gets prices and valuation and looks at price trend, volatility, valuation, and what news explains the moves; the financial analyst gets financials and earnings and looks at growth, margins, debt, liquidity, cash flow, and the last EPS surprise; and the news analyst only gets articles and looks at what each event means for revenue, sentiment, or risk.

`route_articles` is the router. It uses each analyst's "handles" list to send articles to the right place (earnings → financial, market → market, news → news). Irrelevant articles don't match any analyst, so they get dropped.

`analyst_agent` runs one specialist. It gets the data from its tools plus up to five article summaries, and it only runs if it has something to work with. It writes 2–3 short findings about what the data means for an investor, and lessons from earlier runs go into its prompt.

In [ ]:
# the three specialist analysts: what each one focuses on, which tools' data
# it gets, and which article categories the router sends to it
SPECIALISTS = {
    "market": {
        "focus": "price trend, volatility and drawdown, valuation, and which "
                 "news explains the price moves",
        "tools": ["prices", "valuation"],
        "handles": ["market"],
    },
    "financial": {
        "focus": "revenue growth, margins, debt, liquidity and free cash "
                 "flow, plus the last earnings surprise and any earnings news",
        "tools": ["financials", "earnings"],
        "handles": ["earnings"],
    },
    "news": {
        "focus": "what each event means for revenue, sentiment or risk",
        "tools": [],
        "handles": ["news"],
    },
}


# router: every article goes to the analyst that handles its category
# (irrelevant ones don't match anybody, so they just get dropped)
def route_articles(articles):
    routed = {name: [] for name in SPECIALISTS}
    for a in articles:
        for name, spec in SPECIALISTS.items():
            if a["category"] in spec["handles"]:
                routed[name].append(a)
    return routed


# runs one specialist: 2-3 findings from its tool data and up to 5 article summaries
def analyst_agent(name, metrics, articles, lessons):
    news = [a["summary"] for a in articles[:5]]
    prompt = (
        f"You're the {name} analyst on a stock research team. "
        f"Focus on {SPECIALISTS[name]['focus']}. "
        "Using only the data below, write 2-3 short findings about what it means for an investor.\n\n"
        f"Metrics: {json.dumps(metrics)}\nNews: {news}\n\n"
        f"Things earlier reports missed: {lessons or 'nothing yet'}\n"
        "Reply in JSON with a 'findings' key (a list of strings)."
    )
    return call_llm(prompt)["findings"]

## Writer

The writer agent receives all the findings plus the company profile, metrics, and article summaries and produces a short report with five sections: overview, price behavior, financial health, recent news, and conclusion. It is told to only use numbers that appear in the data, and to say so in one sentence when a section has no data (for example if the planner skipped that tool).

In [ ]:
# writer agent - turns all the findings into the report
def writer_agent(ticker, findings, data, lessons):
    prompt = (
        f"Write a short research report on {ticker} with these sections: "
        "overview, price behavior, financial health, recent news, and conclusion. "
        "Base it on the analyst findings and only use numbers that show up in the data. "
        "If a section has no data, say that in one sentence instead of guessing.\n"
        f"Things earlier reports missed (make sure this one covers them): {lessons or 'nothing yet'}\n\n"
        f"Findings: {json.dumps(findings)}\n\n"
        f"Data: {json.dumps(data)}\n\n"
        "Reply in JSON with a 'report' key that holds the whole report as one string, "
        "with the sections separated by blank lines."
    )
    return call_llm(prompt, 0.3)["report"]

## Memory

Implemented by a json file with a list of lessons. After every run we add any check the first draft failed (skipping duplicates). We use the first draft because the final one passes almost everything after revisions (92–100 in our runs).

The checks don't contain ticker-specific numbers, so a lesson like "Says what max_drawdown_pct means for an investor" works for other stocks too. On the next run the lessons get passed into the planner, analyst, and writer prompts. `save_memory=False` skips the saving step, which is how the memory test's no-memory runs stay out of the lessons file.

In [ ]:
# lessons file = just a list of failed checks (lives in /content, so it's gone when the runtime resets)
MEMORY_FILE = "agent_memory.json"

def load_lessons():
    if os.path.exists(MEMORY_FILE):
        with open(MEMORY_FILE) as f:
            return json.load(f)
    return []

# add every check that failed, unless it's already saved
def save_lessons(checks, passed):
    lessons = load_lessons()
    for check, ok in zip(checks, passed):
        if not ok and check not in lessons:
            lessons.append(check)
    with open(MEMORY_FILE, "w") as f:
        json.dump(lessons, f, indent=2)

## Evaluator–Optimizer

Checklist is built inside "run_research": one check per metric plus the general checks below. Our first checklist was too easy (every run got to 92–100 within three rounds, and checks like "points out two risks" or the per-article news checks never failed), so those were swapped for harder ones that test the reasoning: does the buy/hold/sell call follow from the analysis, are the risks backed by numbers, is there a concrete thing to watch, does a headline get tied to a number, and where do the signals disagree. There's also a 400-word limit so the optimizer can't just keep tacking on sentences.

The evaluator agent (`evaluator_agent`) grades the report. Some checks are plain python: the "only uses numbers from the data" check (pulls every $, %, and decimal out of the report and sees if it appears in the data), the fiscal year check (the year of the latest annual statements has to show up in the report), the word limit, and the three checks in `word_checks` that just need certain words (buy/hold/sell in the conclusion, a confidence level, and a time frame). Everything else is a yes/no LLM call to gpt-4o (the one place we use the bigger model), one check at a time. The grader has to copy the sentences that do it first, and a pass only counts if every quote is really in the report (compared on letters and numbers only, so small punctuation changes don't matter).

Score = % of checks passed. Failed checks go to the optimizer agent (`optimizer_agent`) along with the sentences the grader looked at, and it's told to only touch what they ask for. A revision is only kept if the score goes up. The grader isn't perfectly consistent, so a few points either way doesn't mean much.

This loop is also how the agent reflects on its own work: it grades its draft against the checklist and revises whatever failed.

In [ ]:
# the checklist. gpt-4o grades these one at a time, except the last three which are just python
general_checks = [
    "Clearly says if the positives outweigh the risks or the other way around",
    "Uses two different metrics together in one sentence (e.g. free cash flow and current ratio)",
    "Admits what the analysis is missing (e.g. no competitor data)",
    "Explains how the industry affects the outlook instead of just naming it",
    "The conclusion gives the two main reasons for the buy/hold/sell call, and nothing else in the report contradicts it",
    "Gives at least two risks, each backed up by a number from the data (e.g. max drawdown, debt-to-equity)",
    "Names something to watch with a concrete trigger, like a number or the next earnings date",
    "Ties at least one headline to an actual number from the data",
    "Points out where the signals disagree (e.g. strong revenue growth but the stock is falling)",
    "Only uses numbers that are actually in the data",
    "Says which fiscal year the financials are from",
    "Stays under 400 words",
]

# these just look for certain words, so a regex is enough
word_checks = {
    "The conclusion uses the word buy, hold or sell for the call": r"\b(buy|hold|sell)\b",
    "Gives a confidence level (low, medium or high)": r"(low|medium|high)[ -]confidence|confidence[^.]*\b(low|medium|high)\b",
    "Gives a time frame (short term or long term)": r"(short|long)[ -]term",
}
general_checks += list(word_checks)

# no per-metric check for these, they're just labels, dates or raw numbers
skip = ["period", "latest_close_usd", "fiscal_year_end", "revenue_billion_usd", "net_income_billion_usd",
        "balance_sheet_date", "quarter_end", "eps_estimate", "eps_actual"]

In [ ]:
# grades a report against the checklist, returns pass/fail for each check plus the score
def evaluator_agent(report, data, checks):
    # number check: every $, % and decimal in the report has to show up somewhere in the data
    dataNums = [float(n) for n in re.findall(r"\d+\.?\d*", json.dumps(data).replace(",", ""))]
    badNums = []
    for n in re.findall(r"\$\d[\d,]*\.?\d*|\d+\.?\d*%|\d+\.\d+", report):
        if float(n.strip("$%.").replace(",", "")) not in dataNums:
            badNums.append(n)

    # letters and numbers only, so a quote still matches if the punctuation is a bit off
    flat = re.sub(r"[^a-z0-9]", "", report.lower())

    passed, fixes = [], []
    for check in checks:
        fix = check
        if "Only uses numbers" in check:
            ok = not badNums
            fix = f"Remove or correct these numbers, they aren't in the data: {badNums}"
        elif "fiscal year" in check:
            # latest fiscal year has to appear in the report (skipped if there are no financials)
            fin = data["metrics"].get("financials")
            ok = not fin or fin["fiscal_year_end"][:4] in report
        elif "under 400 words" in check:
            ok = len(report.split()) < 400
            fix = f"Cut the report to under 400 words (it has {len(report.split())}) without dropping anything the other checks need"
        elif check in word_checks:
            # the buy/hold/sell one only looks at the conclusion
            text = report.lower()
            if "conclusion" in check:
                text = text.split("conclusion")[-1]
            ok = re.search(word_checks[check], text) is not None
        else:
            # everything else is a yes/no gpt-4o call, one check at a time
            prompt = (
                f"You're grading one thing in a student's stock report: {check}\n\n"
                f"Report:\n{report}\n\n"
                "Only judge whether the report does this, not whether you agree with it. "
                "First copy the sentences that do it exactly as written (an empty list if none), then decide.\n"
                "Reply in JSON with 'quotes' (a list) and 'passed' (true or false)."
            )
            result = call_llm(prompt, 0, "gpt-4o")

            # a pass only counts if every quote is really in the report, and the
            # optimizer gets to see the sentences the grader looked at
            quotes = [q for q in result["quotes"] if re.sub(r"[^a-z0-9]", "", q.lower()) in flat]
            ok = result["passed"] and len(quotes) > 0 and len(quotes) == len(result["quotes"])
            if quotes:
                fix = f"{check} (sentences the grader looked at: {quotes})"

        passed.append(ok)
        if not ok:
            fixes.append(fix)

    return {"passed": passed, "fixes": fixes, "score": round(100 * sum(passed) / len(passed))}


# optimizer: rewrites the report to fix whatever failed
def optimizer_agent(report, fixes, data):
    prompt = (
        "Edit this stock report so it meets every requirement below. "
        "Only change or add the sentences you need to and leave the rest alone. "
        "Only use numbers and facts from the data.\n\n"
        f"Requirements: {fixes}\n\n"
        f"Data: {json.dumps(data)}\n\n"
        f"Report:\n{report}\n\n"
        "Reply in JSON with 'report' (the full edited report as one string) "
        "and 'changes' (a list of what you changed)."
    )
    result = call_llm(prompt)

    # 'changes' is only used for this printout
    print("Changes made:")
    for c in result["changes"]:
        print("  -", c)
    return result["report"]

## Pipeline

`run_research` is the coordinator. It calls each agent in order: planner > tools (the ones the planner picked, plus `get_news` + `preprocess`) > article agent on each headline > router > the analysts that have something to work with > writer > build checklist > evaluator/optimizer loop > save lessons. Prints a lot so you can follow each step.

Everything up to the router is in `gather`. The memory test calls it once per ticker and passes the result into both runs (`inputs=`), so both see the same plan, data, and article summaries.

"max_rounds=0" skips revisions. "use_memory=False" ignores the lessons file, and "save_memory=False" doesn't write to it. `gather(..., show_chain=False)` skips the prompt chain example (the memory test uses it).

In [ ]:
# planner + tools + article agent + router (show_chain=False skips the chain example printout)
def gather(ticker, lessons, show_chain=True):
    stock = yf.Ticker(ticker)
    info = stock.info
    profile = {"name": info["longName"], "sector": info["sector"], "industry": info["industry"]}

    plan = planner_agent(ticker, profile, lessons)
    print(f"Plan: {plan['steps']}, {plan['period']} of prices. {plan['reason']}")

    # run only the tools the planner picked, news always runs
    metrics = {}
    for name in plan["steps"]:
        metrics[name] = run_tool(name, stock, plan)
    texts = preprocess(get_news(stock))

    # prompt chain on every headline, then route each one to its analyst
    articles = [article_agent(text, ticker) for text in texts]
    routed = route_articles(articles)
    relevant = routed["market"] + routed["financial"] + routed["news"]

    print(f"Articles: {len(texts)} | market: {len(routed['market'])}, financial: {len(routed['financial'])}, news: {len(routed['news'])}")
    if relevant and show_chain:
        print("Chain example:")
        for key in ["text", "category", "facts", "summary"]:
            print(f"  {key}: {relevant[0][key]}")

    data = {"company": profile, "metrics": metrics, "articles": [a["summary"] for a in relevant]}
    return data, routed


# coordinator: calls each agent in order. inputs = whatever gather returned,
# so two runs can share the exact same plan and data
def run_research(ticker, max_rounds=3, use_memory=True, save_memory=True, inputs=None):
    lessons = load_lessons() if use_memory else []
    print(f"Researching {ticker}... (using {len(lessons)} lessons from past runs)")
    if inputs is None:
        inputs = gather(ticker, lessons)
    data, routed = inputs
    metrics = data["metrics"]

    # an analyst only runs if it has tool data or articles to work with
    findings = {}
    for name, spec in SPECIALISTS.items():
        toolData = {}
        for t in spec["tools"]:
            if metrics.get(t):
                toolData[t] = metrics[t]
        if toolData or routed[name]:
            findings[name] = analyst_agent(name, toolData, routed[name], lessons)

    report = writer_agent(ticker, findings, data, lessons)

    # one check per metric (except the ones in skip) plus the general ones
    checks = []
    for group in metrics.values():
        for k in group:
            if k not in skip:
                checks.append(f"Says what {k} means for an investor (is it a strength or a concern?)")
    checks += general_checks

    review = evaluator_agent(report, data, checks)
    firstReport, firstReview = report, review
    scores = [review["score"]]
    print(f"\nFirst draft: {sum(review['passed'])}/{len(checks)} checks passed (score {review['score']})")

    # evaluator-optimizer loop, a revision is only kept if it scores higher
    for i in range(1, max_rounds + 1):
        if review["score"] == 100:
            break
        print(f"\nRound {i} fixes:")
        for fix in review["fixes"]:
            print("  -", fix)
        newReport = optimizer_agent(report, review["fixes"], data)
        newReview = evaluator_agent(newReport, data, checks)
        scores.append(newReview["score"])
        print(f"Round {i} score: {review['score']} -> {newReview['score']}")
        if newReview["score"] > review["score"]:
            report, review = newReport, newReview

    # lessons come from the first draft (the final one usually passes almost everything)
    if save_memory:
        save_lessons(checks, firstReview["passed"])

    return {"checks": checks, "scores": scores, "first_report": firstReport, "first_review": firstReview,
            "report": report, "review": review, "lessons": len(lessons)}

## AAPL example

Starts with empty memory. Prints the first draft and the final report, then the checklist showing which checks improved.

In [ ]:
# start with an empty memory file
if os.path.exists(MEMORY_FILE):
    os.remove(MEMORY_FILE)

result = run_research("AAPL")

print(f"\n--- ORIGINAL REPORT (score {result['first_review']['score']}/100) ---\n")
print(result["first_report"])
print(f"\n--- OPTIMIZED REPORT (score {result['review']['score']}/100) ---\n")
print(result["report"])
print("\nChecklist (first draft -> final):")

for check, before, after in zip(result["checks"], result["first_review"]["passed"], result["review"]["passed"]):
    print(f"  {'PASS' if before else 'FAIL'} -> {'PASS' if after else 'FAIL'}  {check[:100]}")

print("\nScores:", result["scores"])

Researching AAPL... (using 0 lessons from past runs)
Plan: ['prices', 'financials', 'valuation'], 1y of prices. AAPL's price behavior over the past year will provide insights into its recent market performance and volatility, while financial health indicators like revenue growth and margins will assess its operational efficiency. Additionally, valuation metrics such as trailing and forward P/E ratios will help determine if the stock is fairly valued relative to its earnings potential, especially in the context of recent market trends and economic conditions.
Articles: 9 | market: 2, financial: 0, news: 4
Chain example:
  text: Apple gets a $360 price target from JP Morgan despite iPhone demand concerns. Just days after Apple’s stock took a hit over reports of iPhone 18 Pro production cuts, JP Morgan has raised its AAPL price target by $20. Here are the details. more…
  category: market
  facts: ['Apple gets a $360 price target from JP Morgan', 'JP Morgan raised its AAPL price target by

## Memory test

Three more tickers (MSFT, JNJ, NVDA). First each one runs without memory, then all three run again with memory. These are full runs (first draft plus up to three revision rounds), so the results show both the first-draft score and how the score changes with each round. Run this right after the AAPL example, since that cell resets memory.

- Without memory, a run doesn't read or write the lessons file, so it's a clean baseline
- With memory, each run reads the lessons saved so far and adds its own. MSFT starts with what AAPL learned, JNJ also gets MSFT's lessons, and NVDA gets both, so lessons from one company get used on the next. None of them see lessons from their own no-memory run
- Each ticker's plan, data, and article summaries are made once up front with no lessons and reused by both of its runs, so the only difference between the two runs is the lessons

The test cell prints the normal run output plus both reports for each run (`show_drafts = False` turns the reports off). The results cell after it prints the scores, like `46 > 79 > 83 > 92` (first draft, then each round), the average first-draft score with and without memory, and the lessons saved by the end.

Caveats: it's one run per ticker per condition and the grader is noisy, so small differences don't mean much. The lessons are basically the checklist items, so the memory run is partly being told what the grader looks for. It shows the lessons carry over to other tickers, not that the reports got better in some general sense.

In [ ]:
# each ticker runs once without memory, then all three run again with memory
tickers = ["MSFT", "JNJ", "NVDA"]
show_drafts = True  # False -> don't print the reports

# plan, data and article summaries get made once per ticker (no lessons) and both runs reuse them
inputs = {}
for t in tickers:
    print(f"\n========== {t}: plan and data ==========")
    inputs[t] = gather(t, [], show_chain=False)

# without memory = doesn't read or write the lessons file
no_mem, with_mem = {}, {}
for t in tickers:
    print(f"\n========== {t} WITHOUT MEMORY ==========")
    no_mem[t] = run_research(t, use_memory=False, save_memory=False, inputs=inputs[t])
    if show_drafts:
        print(f"\n--- ORIGINAL REPORT (score {no_mem[t]['scores'][0]}/100) ---\n\n{no_mem[t]['first_report']}")
        print(f"\n--- FINAL REPORT (score {no_mem[t]['review']['score']}/100) ---\n\n{no_mem[t]['report']}")

# with memory = reads the lessons so far (starting with AAPL's) and adds its own,
# so JNJ also gets MSFT's lessons and NVDA gets both
for t in tickers:
    print(f"\n========== {t} WITH MEMORY ==========")
    with_mem[t] = run_research(t, inputs=inputs[t])
    if show_drafts:
        print(f"\n--- ORIGINAL REPORT (score {with_mem[t]['scores'][0]}/100) ---\n\n{with_mem[t]['first_report']}")
        print(f"\n--- FINAL REPORT (score {with_mem[t]['review']['score']}/100) ---\n\n{with_mem[t]['report']}")


========== MSFT: plan and data ==========
Plan: ['prices', 'financials', 'valuation'], 1y of prices. Using a 1-year period allows us to capture recent price behavior and volatility while also providing a comprehensive view of Microsoft's financial health and valuation metrics. This timeframe is particularly relevant given the dynamic nature of the technology sector and the recent developments in the software industry.
Articles: 9 | market: 1, financial: 0, news: 0

========== JNJ: plan and data ==========
Plan: ['prices', 'financials', 'earnings'], 1y of prices. A 1-year period provides a balanced view of Johnson & Johnson's recent price behavior, financial health, and earnings performance, which are crucial for understanding the company's current market position and potential future performance. This timeframe captures both short-term volatility and longer-term trends, making it suitable for a comprehensive analysis.
Articles: 10 | market: 3, financial: 0, news: 2

========== NVDA: p

In [ ]:
# scores after each round, first draft first. if a number doesn't go up, that revision got
# thrown out and the next round tried again from the better version
first_off, first_on = 0, 0
for t in tickers:
    print(t)
    print("  without memory:", " > ".join(str(s) for s in no_mem[t]["scores"]))
    print("  with memory:   ", " > ".join(str(s) for s in with_mem[t]["scores"]), f"(used {with_mem[t]['lessons']} lessons)")
    first_off += no_mem[t]["scores"][0]
    first_on += with_mem[t]["scores"][0]

print(f"\nAverage first draft: {first_off / len(tickers):.0f} without memory, {first_on / len(tickers):.0f} with memory")
print("\nLessons in memory now (AAPL's plus what the memory runs added):")
for lesson in load_lessons():
    print("  -", lesson)

MSFT
  without memory: 56 > 96 > 96 > 92
  with memory:    88 > 96 > 96 > 92 (used 12 lessons)
JNJ
  without memory: 46 > 79 > 83 > 92
  with memory:    83 > 92 > 92 > 92 (used 14 lessons)
NVDA
  without memory: 60 > 100
  with memory:    96 > 96 > 96 > 96 (used 16 lessons)

Average first draft: 54 without memory, 89 with memory

Lessons in memory now (AAPL's plus what the memory runs added):
  - Says what revenue_growth_pct means for an investor (is it a strength or a concern?)
  - Says what trailing_pe means for an investor (is it a strength or a concern?)
  - Says what forward_pe means for an investor (is it a strength or a concern?)
  - Clearly says if the positives outweigh the risks or the other way around
  - Uses two different metrics together in one sentence (e.g. free cash flow and current ratio)
  - Admits what the analysis is missing (e.g. no competitor data)
  - Explains how the industry affects the outlook instead of just naming it
  - Names something to watch with a conc

## Conclusion / findings

The agent reliably produces a short research report that covers price behavior, financial health, and recent news, then improves it against a fixed checklist. The planner picks its own tools for each company based on a quick profile. On the AAPL example the first draft scored 52 and the final version 92 after 2 revision rounds (the third round didn't beat it, so it wasn't kept). In the memory test (each ticker once without memory and once with it, lessons carrying over from one ticker to the next) the average first-draft score went from 54 without lessons to 89 with them (MSFT 56 → 88, JNJ 46 → 83, NVDA 60 → 96), showing that the saved checks transfer across tickers.

A few practical limits showed up. The first checklist was too easy (every run got to 92–100 within three rounds and several checks never failed), so it was replaced with harder checks about the reasoning. With the new checklist the first drafts without memory scored 46–60, but every run still ended between 92 and 100 after revisions, and three checks (the two reasons behind the call, risks backed by numbers, and only using numbers from the data) never failed once. The grader is somewhat noisy (the same check can pass or fail on nearly identical text), so small score changes don't mean much. Grading the word checks with regex and making the grader quote the report helps, but the remaining checks are still LLM calls. The lessons are essentially the checklist items themselves, which means the memory benefit is partly "the model is told what will be graded" rather than a general improvement in reasoning. The tools also assume a normal company with the usual statement lines. If one of them fails (a bank's balance sheet has no current assets, for example) the run keeps going without it, but ETFs still won't work since they have no sector or industry for the profile.

Overall the pipeline works for the intended use (quick, checklist-driven research notes on regular stocks) and the memory mechanism does what it was designed to do.